In [ ]:
# IL-only GPU FORWARD V6: SDPA + LOSS (JIT error log) - 1/6: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
!nvidia-smi
r=subprocess.run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mono-complete', shell=True, capture_output=True, text=True, timeout=1200)
print('mono install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('mono --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# 2/3: clone backend + symlink cudart/cublas/cuda(driver)
import os, subprocess, glob
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
libdir='/content/tkvlib'; os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        found=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if found: return sorted(found)[0]
    return None
cudart=pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*'])
cublas=pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*'])
cudrv=pick(['/usr/lib64-nvidia/libcuda.so*','/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
print('cudart=',cudart); print('cublas=',cublas); print('cuda driver=',cudrv)
for nm,tgt in [('cudart.dll',cudart),('cublas.dll',cublas),('cuda.dll',cudrv)]:
    if tgt is None: continue
    p=libdir+'/'+nm
    if os.path.lexists(p): os.remove(p)
    os.symlink(tgt,p); print('symlink',nm,'->',tgt)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH=',os.environ['LD_LIBRARY_PATH'][:160])

In [ ]:
# 3/4: chay SDPA
import subprocess, os
exe=d+'/Release/cuda_backend/examples/ilgpu_sdpa.exe'
print('exe exists=',os.path.exists(exe),' bytes=',os.path.getsize(exe) if os.path.exists(exe) else 0)
print('co C++/CUDA sinh?', [p for p in ['ilgpu_sdpa.torch.cpp','ilgpu_sdpa.cuda.cu'] if os.path.exists(os.path.dirname(exe)+'/'+p)])
r=subprocess.run(['mono',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono rc=',r.returncode)
print(r.stdout)
print('STDERR:',(r.stderr or '')[:1500])

In [ ]:
# 4/4: reference Python thuan
import re, math
T,D=4,8
Q=[[1.0+0.1*(i+j) for j in range(D)] for i in range(T)]
K=[[0.5+0.05*(i-j) for j in range(D)] for i in range(T)]
V=[[0.2+0.03*(i+j) for j in range(D)] for i in range(T)]
S=[[sum(Q[i][d]*K[j][d] for d in range(D))*0.35355339 for j in range(T)] for i in range(T)]
for i in range(T):
    for j in range(T):
        if j>i: S[i][j]=-1e9
P=[]
for i in range(T):
    e=[math.exp(v) for v in S[i]]; z=sum(e); P.append([v/z for v in e])
O=[[sum(P[i][j]*V[j][d] for j in range(T)) for d in range(D)] for i in range(T)]
got={}
for a,b,v in re.findall(r'^OUT\s+(\d+)\s+(\d+)\s+([0-9.eE+-]+)$',r.stdout,re.M): got[(int(a),int(b))]=float(v)
errs=[abs(got[(i,j)]-O[i][j]) for i in range(T) for j in range(D) if (i,j) in got]
print('n=',len(errs),' maxerr=', max(errs) if errs else None)
print('KET LUAN:', 'OK' if len(errs)==T*D and max(errs)<2e-4 else 'XEM OUTPUT')

In [ ]:
# 5/6: chay LOSS kernel (log-softmax + gather + mean)
import subprocess, os, re
exe3=d+'/Release/cuda_backend/examples/ilgpu_loss.exe'
print('loss exe exists=',os.path.exists(exe3),' bytes=',os.path.getsize(exe3) if os.path.exists(exe3) else 0)
r3=subprocess.run(['mono',exe3], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono loss rc=',r3.returncode)
print(r3.stdout)
print('STDERR:',(r3.stderr or '')[:1200])

In [ ]:
# 6/6: reference loss Python thuan
import re, math
T,V=4,8
x=[[(i-j)*0.25 for j in range(V)] for i in range(T)]
lp=[]
for row in x:
    m=max(row); e=[math.exp(v-m) for v in row]; s=sum(e)
    lp.append([v-m-math.log(s) for v in row])
ref=[-sum(lp[i][(i+1)%V] for i in range(T))/T]
got=[float(v) for v in re.findall(r'^LOSS\s+([0-9.eE+-]+)$',r3.stdout,re.M)]
print('LOSS ref=',ref[0],' got=',got[0] if got else None,' err=',abs(got[0]-ref[0]) if got else None)
print('KET LUAN LOSS:', 'OK' if got and abs(got[0]-ref[0])<2e-4 else 'XEM OUTPUT')